# Circuitos Electorales de Argentina
## Notebook para explorar los GeoJSON del repositorio `circuitos_electorales_AR`.
## Genera una tabla con los circuitos por provincia y un resumen de totales.


In [1]:
import json
import glob
import pandas as pd
from pathlib import Path
from IPython.display import display

In [2]:

# ──────────────────────────────────────────────
# Configuración — ajustá este path si es necesario
# ──────────────────────────────────────────────
GEOJSON_DIR = Path("geojson")   # relativo a donde esté el notebook dentro del repo

In [3]:

# Mapa codprov → nombre de provincia (orden oficial Argentina)
PROVINCIAS = {
    "01": "Ciudad Autónoma de Buenos Aires",
    "02": "Buenos Aires",
    "03": "Catamarca",
    "04": "Córdoba",
    "05": "Corrientes",
    "06": "Chaco",
    "07": "Chubut",
    "08": "Entre Ríos",
    "09": "Formosa",
    "10": "Jujuy",
    "11": "La Pampa",
    "12": "La Rioja",
    "13": "Mendoza",
    "14": "Misiones",
    "15": "Neuquén",
    "16": "Río Negro",
    "17": "Salta",
    "18": "San Juan",
    "19": "San Luis",
    "20": "Santa Cruz",
    "21": "Santa Fe",
    "22": "Santiago del Estero",
    "23": "Tucumán",
    "24": "Tierra del Fuego",
}

# Mapa nombre de archivo (sin extensión, uppercase) → codprov
# Cubre variantes de nombre que pueden aparecer en el repo
ARCHIVO_A_CODPROV = {
    "CABA":                      "01",
    "CIUDAD_AUTONOMA_DE_BUENOS_AIRES": "01",
    "BUENOS_AIRES":              "02",
    "CATAMARCA":                 "03",
    "CORDOBA":                   "04",
    "CORRIENTES":                "05",
    "CHACO":                     "06",
    "CHUBUT":                    "07",
    "ENTRE_RIOS":                "08",
    "FORMOSA":                   "09",
    "JUJUY":                     "10",
    "LA_PAMPA":                  "11",
    "LAPAMPA":                   "11",
    "LA_RIOJA":                  "12",
    "MENDOZA":                   "13",
    "MISIONES":                  "14",
    "NEUQUEN":                   "15",
    "RIO_NEGRO":                 "16",
    "SALTA":                     "17",
    "SAN_JUAN":                  "18",
    "SAN_LUIS":                  "19",
    "SANTA_CRUZ":                "20",
    "SANTA_FE":                  "21",
    "SANTIAGO_DEL_ESTERO":       "22",
    "TUCUMAN":                   "23",
    "TIERRA_DEL_FUEGO":          "24",
}

In [4]:
def codprov_desde_archivo(nombre_archivo: str) -> str:
    """Infiere el codprov a partir del nombre del archivo GeoJSON."""
    clave = Path(nombre_archivo).stem.upper()
    return ARCHIVO_A_CODPROV.get(clave, None)


In [5]:
# %%
# ──────────────────────────────────────────────
# 1. Listar archivos GeoJSON locales
# ──────────────────────────────────────────────
archivos = sorted(GEOJSON_DIR.glob("*.geojson"))
print(f"Archivos encontrados: {len(archivos)}")
for f in archivos:
    print(f"  {f.name}")


Archivos encontrados: 24
  CABA.geojson
  CATAMARCA.geojson
  CHACO.geojson
  CHUBUT.geojson
  CORDOBA.geojson
  CORRIENTES.geojson
  ENTRERIOS.geojson
  FORMOSA.geojson
  JUJUY.geojson
  LAPAMPA.geojson
  LA_RIOJA.geojson
  MENDOZA.geojson
  MISIONES.geojson
  NEUQUEN.geojson
  PBA.geojson
  RIO_NEGRO.geojson
  SALTA.geojson
  SANLUIS.geojson
  SANTACRUZ.geojson
  SANTAFE.geojson
  SANTIAGO_DEL_ESTERO.geojson
  SAN_JUAN.geojson
  TIERRA_DEL_FUEGO.geojson
  TUCUMAN.geojson


In [6]:

# ──────────────────────────────────────────────
# 2. Leer todos los GeoJSON e iterar features
# ──────────────────────────────────────────────
registros = []

for archivo in archivos:
    with open(archivo, encoding="utf-8") as f:
        gj = json.load(f)

    for feature in gj.get("features", []):
        props = feature.get("properties", {})

        # Normalizar nombres de columnas (por si hay variaciones de mayúsculas)
        props_lower = {k.lower(): v for k, v in props.items()}

        circuito = props_lower.get("circuito")
        coddepto = str(props_lower.get("coddepto", "")).zfill(3)

        # Prioridad: codprov del archivo → fallback al nombre del archivo
        _raw = props_lower.get("codprov")
        codprov_prop = str(_raw).zfill(2) if _raw not in (None, "", "0", 0) else "00"
        codprov = (
            codprov_prop
            if codprov_prop not in ("00", "")
            else codprov_desde_archivo(archivo.name) or "??"
        )

        registros.append({
            "archivo":   archivo.name,
            "codprov":   codprov,
            "provincia": PROVINCIAS.get(codprov, f"Desconocida ({codprov})"),
            "coddepto":  coddepto,
            "circuito":  circuito,
        })

df = pd.DataFrame(registros)
print(f"Total de circuitos cargados: {len(df)}")

# Diagnóstico: archivos que no pudieron resolverse
sin_prov = df[df["codprov"] == "??"]["archivo"].unique()
if len(sin_prov):
    print(f"\n⚠️  Archivos sin provincia resuelta — agregá su clave al diccionario ARCHIVO_A_CODPROV:")
    for f in sin_prov:
        print(f"  '{Path(f).stem.upper()}': '??'  ← reemplazá ?? por el codprov correcto")



Total de circuitos cargados: 5589


In [7]:
# %%
# ──────────────────────────────────────────────
# 3. Tabla detallada por provincia
# ──────────────────────────────────────────────
print("=" * 60)
print("CIRCUITOS POR PROVINCIA — DETALLE")
print("=" * 60)

for prov_nombre, grupo in df.groupby("provincia", sort=True):
    circuitos = sorted(grupo["circuito"].dropna().unique().tolist())
    tabla = pd.DataFrame({
        "#":        range(1, len(circuitos) + 1),
        "Circuito": circuitos,
    }).set_index("#")

    print(f"\n{'─'*50}")
    print(f"  {prov_nombre}  (codprov: {grupo['codprov'].iloc[0]})")
    print(f"  Total circuitos: {len(circuitos)}")
    print(f"{'─'*50}")
    display(tabla)

CIRCUITOS POR PROVINCIA — DETALLE

──────────────────────────────────────────────────
  Buenos Aires  (codprov: 02)
  Total circuitos: 1066
──────────────────────────────────────────────────


,Circuito
#,
1,00001
2,00002
3,00003
4,00004
5,00005
...,...
1062,0973A
1063,0975A
1064,0975B



──────────────────────────────────────────────────
  Catamarca  (codprov: 03)
  Total circuitos: 155
──────────────────────────────────────────────────


,Circuito
#,
1,00001
2,00002
3,00003
4,00004
5,00005
...,...
151,00151
152,00152
153,00153



──────────────────────────────────────────────────
  Chaco  (codprov: 06)
  Total circuitos: 153
──────────────────────────────────────────────────


...

,Circuito
#,
1,00001
2,00002
3,00003
4,00004
5,00004A
...,...
253,0190A
254,0193A
255,0195A



──────────────────────────────────────────────────
  Tierra del Fuego  (codprov: 24)
  Total circuitos: 25
──────────────────────────────────────────────────


,Circuito
#,
1,0004
2,0008
3,0010
4,0011
5,0012
6,0013
7,0014
8,0016
9,0018



──────────────────────────────────────────────────
  Tucumán  (codprov: 23)
  Total circuitos: 274
──────────────────────────────────────────────────


,Circuito
#,
1,00001
2,00002
3,00003
4,00004
5,00005
...,...
270,0216A
271,0223A
272,0223B


In [8]:
# %%
# ──────────────────────────────────────────────
# 4. Tabla resumen: cantidad de circuitos por provincia
# ──────────────────────────────────────────────
resumen = (
    df.groupby(["codprov", "provincia"])["circuito"]
    .nunique()
    .reset_index()
    .rename(columns={"circuito": "total_circuitos"})
    .sort_values("codprov")
    .reset_index(drop=True)
)
resumen.index += 1

print("\n" + "=" * 60)
print("RESUMEN: TOTAL DE CIRCUITOS POR PROVINCIA")
print("=" * 60)
display(resumen)

print(f"\nTotal provincias/distritos: {len(resumen)}")
print(f"Total circuitos (país):     {resumen['total_circuitos'].sum()}")


RESUMEN: TOTAL DE CIRCUITOS POR PROVINCIA


,codprov,provincia,total_circuitos
1,01,Ciudad Autónoma de Buenos Aires,167
2,02,Buenos Aires,1066
3,03,Catamarca,155
4,04,Córdoba,626
5,05,Corrientes,170
6,06,Chaco,153
7,07,Chubut,115
8,08,Entre Ríos,318
9,09,Formosa,107
10,10,Jujuy,146



Total provincias/distritos: 24
Total circuitos (país):     5557


In [9]:
# ──────────────────────────────────────────────
# 5. (Opcional) Exportar a CSV
# ──────────────────────────────────────────────
df.to_csv("circuitos_detalle.csv", index=False)
resumen.to_csv("circuitos_resumen.csv", index=False)
print("Archivos exportados: circuitos_detalle.csv y circuitos_resumen.csv")

Archivos exportados: circuitos_detalle.csv y circuitos_resumen.csv
